# SQL 练习 Notebook

这份 notebook 用来练习 `JOIN` 和子查询。

使用方式：
1. 从上到下依次运行。
2. 先运行“初始化数据库”代码块。
3. 在练习题代码块里把 `your_sql` 改成你自己的 SQL。
4. 运行后观察结果，再和后面的参考答案对照。

这里使用的是 Python 自带的 `sqlite3`，所以不需要终端也可以练。

## 1. 初始化数据库

Goal：创建练习用的内存数据库，并插入测试数据。

Pitfalls：如果你重启了 kernel，需要重新运行这一部分。

In [16]:
import sqlite3

conn = sqlite3.connect(':memory:')
cur = conn.cursor()

setup_sql = '''
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    city TEXT
);

CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT
);

CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY,
    emp_name TEXT,
    salary REAL,
    dept_id INTEGER
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    amount REAL,
    order_date TEXT
);

INSERT INTO customers VALUES
(1, 'Alice', 'Shanghai'),
(2, 'Bob', 'Beijing'),
(3, 'Cindy', 'Shanghai'),
(4, 'David', 'Shenzhen'),
(5, 'Eric', 'Hangzhou');

INSERT INTO departments VALUES
(10, 'Sales'),
(20, 'IT'),
(30, 'HR');

INSERT INTO employees VALUES
(101, 'Tom', 8000, 10),
(102, 'Jerry', 12000, 20),
(103, 'Lily', 9000, 10),
(104, 'Lucy', 15000, 20),
(105, 'Ben', 7000, 30),
(106, 'Anna', 9500, 30);

INSERT INTO orders VALUES
(1001, 1, 300, '2024-01-10'),
(1002, 1, 500, '2024-02-15'),
(1003, 2, 800, '2024-01-20'),
(1004, 3, 200, '2024-03-01'),
(1005, 3, 900, '2024-03-18'),
(1006, 4, 400, '2024-02-28');
'''

cur.executescript(setup_sql)
conn.commit()
print('数据库初始化完成，可以开始练习。')

数据库初始化完成，可以开始练习。


In [17]:
def run_sql(sql):
    cursor = conn.execute(sql)
    columns = [desc[0] for desc in cursor.description] if cursor.description else []
    rows = cursor.fetchall()

    if not columns:
        print('语句执行完成，没有返回结果集。')
        return

    print(' | '.join(columns))
    print('-' * 60)
    for row in rows:
        print(' | '.join(str(value) for value in row))

print('辅助函数已就绪，后面直接调用 run_sql(your_sql) 即可。')

辅助函数已就绪，后面直接调用 run_sql(your_sql) 即可。


## 2. 先看原始数据

Step-by-step：先看表和数据，后面写 `JOIN` 时才知道该连谁。

In [18]:
run_sql('SELECT * FROM customers;')

customer_id | customer_name | city
------------------------------------------------------------
1 | Alice | Shanghai
2 | Bob | Beijing
3 | Cindy | Shanghai
4 | David | Shenzhen
5 | Eric | Hangzhou


In [19]:
run_sql('SELECT * FROM orders;')

order_id | customer_id | amount | order_date
------------------------------------------------------------
1001 | 1 | 300.0 | 2024-01-10
1002 | 1 | 500.0 | 2024-02-15
1003 | 2 | 800.0 | 2024-01-20
1004 | 3 | 200.0 | 2024-03-01
1005 | 3 | 900.0 | 2024-03-18
1006 | 4 | 400.0 | 2024-02-28


In [20]:
run_sql('SELECT * FROM departments;')

dept_id | dept_name
------------------------------------------------------------
10 | Sales
20 | IT
30 | HR


In [21]:
run_sql('SELECT * FROM employees;')

emp_id | emp_name | salary | dept_id
------------------------------------------------------------
101 | Tom | 8000.0 | 10
102 | Jerry | 12000.0 | 20
103 | Lily | 9000.0 | 10
104 | Lucy | 15000.0 | 20
105 | Ben | 7000.0 | 30
106 | Anna | 9500.0 | 30


## 3. JOIN 练习

建议流程：
- 先自己写
- 运行看结果
- 再看参考答案

注意：普通 SQL 语句最后要加分号。

### 题目 1

Goal：查询每个订单对应的客户姓名、城市、订单金额。

In [28]:
your_sql = '''
SELECT customer_name ,city ,orders.amount
FROM customers
INNER JOIN orders
ON customers.customer_id = orders.customer_id;
'''

run_sql(your_sql)

customer_name | city | amount
------------------------------------------------------------
Alice | Shanghai | 300.0
Alice | Shanghai | 500.0
Bob | Beijing | 800.0
Cindy | Shanghai | 200.0
Cindy | Shanghai | 900.0
David | Shenzhen | 400.0


In [29]:
answer_sql = '''
SELECT
    o.order_id,
    c.customer_name,
    c.city,
    o.amount
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id;
'''

run_sql(answer_sql)

order_id | customer_name | city | amount
------------------------------------------------------------
1001 | Alice | Shanghai | 300.0
1002 | Alice | Shanghai | 500.0
1003 | Bob | Beijing | 800.0
1004 | Cindy | Shanghai | 200.0
1005 | Cindy | Shanghai | 900.0
1006 | David | Shenzhen | 400.0


### 题目 2

Goal：查询所有员工姓名、工资，以及所属部门名称。

In [31]:
your_sql = '''
SELECT e.emp_name , e.salary ,d.dept_name
FROM employees as e
LEFT JOIN departments as d
ON e.dept_id = d.dept_id
'''

run_sql(your_sql)

emp_name | salary | dept_name
------------------------------------------------------------
Tom | 8000.0 | Sales
Jerry | 12000.0 | IT
Lily | 9000.0 | Sales
Lucy | 15000.0 | IT
Ben | 7000.0 | HR
Anna | 9500.0 | HR


In [33]:
answer_sql = '''
SELECT
    e.emp_name,
    e.salary,
    d.dept_name
FROM employees e
JOIN departments d
    ON e.dept_id = d.dept_id;
'''

run_sql(answer_sql)

emp_name | salary | dept_name
------------------------------------------------------------
Tom | 8000.0 | Sales
Jerry | 12000.0 | IT
Lily | 9000.0 | Sales
Lucy | 15000.0 | IT
Ben | 7000.0 | HR
Anna | 9500.0 | HR


### 题目 3

Goal：查询没有下过单的客户。

Hint：想一想 `LEFT JOIN` 后哪些行没有匹配成功。

In [36]:
your_sql = '''
SELECT customer_name , amount
FROM customers
LEFT JOIN orders
ON customers.customer_id = orders.customer_id
WHERE amount IS NULL;
'''

run_sql(your_sql)

customer_name | amount
------------------------------------------------------------
Eric | None


In [37]:
answer_sql = '''
SELECT
    c.customer_id,
    c.customer_name
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;
'''

run_sql(answer_sql)

customer_id | customer_name
------------------------------------------------------------
5 | Eric


### 题目 4

Goal：查询每个客户的下单总金额。

Key APIs：`JOIN`、`SUM()`、`GROUP BY`。

In [42]:
your_sql = '''
SELECT c.customer_name , SUM(amount)
FROM customers AS c
LEFT JOIN orders AS o
ON c.customer_id = o.customer_id
GROUP BY c.customer_id , c.customer_name;
'''

run_sql(your_sql)

customer_name | SUM(amount)
------------------------------------------------------------
Alice | 800.0
Bob | 800.0
Cindy | 1100.0
David | 400.0
Eric | None


In [41]:
answer_sql = '''
SELECT
    c.customer_name,
    SUM(o.amount) AS total_amount
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name;
'''

run_sql(answer_sql)

customer_name | total_amount
------------------------------------------------------------
Alice | 800.0
Bob | 800.0
Cindy | 1100.0
David | 400.0


## 4. 子查询练习

### 题目 5

Goal：查询工资高于公司平均工资的员工。

In [50]:
your_sql = '''
SELECT 
    *
FROM
    employees as e
WHERE e.salary > (
SELECT AVG(salary) 
FROM employees
);
'''

run_sql(your_sql)

emp_id | emp_name | salary | dept_id
------------------------------------------------------------
102 | Jerry | 12000.0 | 20
104 | Lucy | 15000.0 | 20


In [51]:
answer_sql = '''
SELECT
    emp_name,
    salary
FROM employees
WHERE salary > (
    SELECT AVG(salary)
    FROM employees
);
'''

run_sql(answer_sql)

emp_name | salary
------------------------------------------------------------
Jerry | 12000.0
Lucy | 15000.0


### 题目 6

Goal：查询订单金额大于平均订单金额的订单。

In [52]:
your_sql = '''
SELECT
*
FROM 
orders
WHERE
amount > (SELECT AVG(amount) FROM orders)
;
'''

run_sql(your_sql)

order_id | customer_id | amount | order_date
------------------------------------------------------------
1003 | 2 | 800.0 | 2024-01-20
1005 | 3 | 900.0 | 2024-03-18


In [53]:
answer_sql = '''
SELECT
    order_id,
    customer_id,
    amount
FROM orders
WHERE amount > (
    SELECT AVG(amount)
    FROM orders
);
'''

run_sql(answer_sql)

order_id | customer_id | amount
------------------------------------------------------------
1003 | 2 | 800.0
1005 | 3 | 900.0


### 题目 7

Goal：查询至少下过 2 次单的客户姓名。

Hint：子查询里先按 `customer_id` 分组统计。

In [65]:
your_sql = '''
SELECT c.customer_id, c.customer_name
FROM customers AS c
JOIN orders AS o
ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
HAVING COUNT(*) > 1;
'''

run_sql(your_sql)

customer_id | customer_name
------------------------------------------------------------
1 | Alice
3 | Cindy


In [66]:
answer_sql = '''
SELECT customer_name
FROM customers
WHERE customer_id IN (
    SELECT customer_id
    FROM orders
    GROUP BY customer_id
    HAVING COUNT(*) >= 2
);
'''

run_sql(answer_sql)

customer_name
------------------------------------------------------------
Alice
Cindy


### 题目 8

Goal：查询下单总金额最高的客户。

Pitfalls：这一题会用到“先汇总，再到外层取最大值”的思路。

In [71]:
your_sql = '''
SELECT c.customer_id , c.customer_name
FROM customers AS c
JOIN (
SELECT customer_id ,sum(amount) as total_amount
FROM orders
GROUP BY customer_id) AS t
ON c.customer_id = t.customer_id
WHERE t.total_amount = (
SELECT MAX(total_amount)
FROM (SELECT SUM(amount) AS total_amount
FROM orders
GROUP BY customer_id)
AS x
);
'''

run_sql(your_sql)

customer_id | customer_name
------------------------------------------------------------
3 | Cindy


In [72]:
answer_sql = '''
SELECT
    t.customer_name,
    t.total_amount
FROM (
    SELECT
        c.customer_name,
        SUM(o.amount) AS total_amount
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.customer_name
) t
WHERE t.total_amount = (
    SELECT MAX(total_amount)
    FROM (
        SELECT SUM(amount) AS total_amount
        FROM orders
        GROUP BY customer_id
    ) x
);
'''

run_sql(answer_sql)

customer_name | total_amount
------------------------------------------------------------
Cindy | 1100.0
